[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-2_executed.ipynb)

---


# 87. The Hub-and-Spoke Network Design Problem

## Tier 2: A Geographic Clustering Heuristic

### Purpose and scope

This notebook implements a heuristic for hub-and-spoke network design. It first partitions locations geographically with K-means, selects one candidate hub within each cluster by enumerating the cluster's nodes, and then accounts for traffic between clusters by routing it through their selected hubs. The method is a heuristic: the geographic partition is fixed before hub selection, so the procedure does not guarantee a globally optimal network.

The input is a set of node coordinates, directed origin-destination demand, distances, hub establishment costs, hub capacity limits, and an inter-hub discount factor. The output is a clustering, a feasible hub selection if one exists, spoke assignments, and a cost breakdown. The example also checks capacity feasibility explicitly; it does not silently assign a hub when none meets the stated capacity rule.

### Modeling assumptions

- Each node belongs to exactly one geographic cluster, and each cluster has exactly one hub located at one of its nodes.
- A flow within a cluster travels from its origin to that cluster's hub, then from the hub to its destination. The second leg is multiplied by the discount factor \(\alpha\).
- A flow between two clusters travels from its origin to its own cluster's hub, between the two hubs, and from the destination cluster's hub to its destination. The two hub-related legs are multiplied by \(\alpha\), matching the cost convention used by this method.
- A candidate hub's capacity is compared with the total directed demand between distinct nodes in its cluster. Inter-cluster traffic is included in routing cost but is not included in this capacity test. This is the capacity rule implemented by the cluster-level selection procedure; it should not be interpreted as a general physical throughput model.
- Distances are Euclidean distances between the supplied coordinates, multiplied by a fixed scale factor. Demand is directed, so the flow from node \(i\) to node \(j\) need not equal the flow from \(j\) to node \(i\).
- Fixed hub costs are charged once for each selected hub. No other facility, transport, or service constraints are modeled.

### Method and interpretation

A *cluster* is a subset of nodes grouped by geographic proximity. A *hub candidate* is a node in that cluster considered as the cluster's facility. Enumerating candidates is manageable within small clusters, while geographic clustering determines which candidate sets are examined. This division of the work is the heuristic's central idea: use geography to break the network into smaller decisions, then connect the resulting hubs.

For cluster \(c\), let \(C_c\) be its nodes and \(h\in C_c\) a candidate hub. The within-cluster flow volume and the cost of selecting that hub are

$$
F_c = \sum_{i\in C_c}\sum_{j\in C_c,\,j\ne i} w_{ij},
\qquad
H_c(h) = f_h + \sum_{i\in C_c}\sum_{j\in C_c,\,j\ne i}
w_{ij}\bigl(d_{ih}+\alpha d_{hj}\bigr).
$$

Here, \(w_{ij}\) is directed demand, \(d_{ij}\) is distance, and \(f_h\) is the fixed cost of opening hub \(h\). A candidate is feasible when \(F_c\le q_h\), where \(q_h\) is its capacity limit. The chosen hub minimizes \(H_c(h)\) among feasible candidates. In the code, these quantities are calculated by `find_optimal_cluster_hub`.

For distinct clusters \(a\) and \(b\), with selected hubs \(h_a\) and \(h_b\), the cost of traffic in both directions is

$$
\begin{aligned}
I_{ab} ={}& \sum_{i\in C_a}\sum_{j\in C_b} w_{ij}
  \bigl(d_{i h_a}+\alpha d_{h_a h_b}+\alpha d_{h_b j}\bigr)\\
 &+\sum_{i\in C_b}\sum_{j\in C_a} w_{ij}
  \bigl(d_{i h_b}+\alpha d_{h_b h_a}+\alpha d_{h_a j}\bigr).
\end{aligned}
$$

The total network cost is the sum of selected hubs' fixed costs, within-cluster variable costs, and inter-cluster routing costs. The implementation uses the same routing rule in `calculate_inter_cluster_cost` and adds each unordered cluster pair once.

K-means assigns nodes to the nearest cluster center under squared Euclidean distance; its result is a geographic partition, not a proof that the partition minimizes network cost. If there are \(n\) nodes, \(k\) clusters, \(I\) K-means iterations, and cluster sizes \(|C_c|\), the main work is approximately \(O(I n k + \sum_c |C_c|^3 + n^2)\) for the implementation below: candidate evaluation considers each candidate and all directed within-cluster pairs, and inter-cluster routing accounts for cross-cluster pairs. Actual runtime also depends on implementation and hardware. This is not generally an \(O(n\log n)\) algorithm.

### What to examine

- Whether K-means forms geographically coherent groups.
- Whether each cluster has at least one capacity-feasible hub candidate.
- How fixed, within-cluster, and inter-cluster costs contribute to a feasible network's total cost.
- How results change across the supplied small-instance and scalability experiments.

The example uses eight nodes. Its capacity limits are assessed using the cluster-level rule stated above; if a chosen clustering has no feasible candidate in a cluster, the notebook reports that no feasible solution was produced under this rule.

In [ ]:
from itertools import combinations
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class DivideConquerHubSpoke:
    """Geographically cluster nodes, select cluster hubs, and price routes."""

    def __init__(
        self,
        coordinates,
        demand_matrix,
        distance_matrix,
        hub_costs,
        capacity_limits,
        alpha=0.75,
        max_hubs=3,
    ):
        self.coordinates = np.asarray(coordinates, dtype=float)
        self.demand_matrix = np.asarray(demand_matrix, dtype=float)
        self.distance_matrix = np.asarray(distance_matrix, dtype=float)
        self.hub_costs = np.asarray(hub_costs, dtype=float)
        self.capacity_limits = np.asarray(capacity_limits, dtype=float)
        self.alpha = float(alpha)
        self.max_hubs = int(max_hubs)
        self.num_nodes = len(self.coordinates)

        expected_matrix_shape = (self.num_nodes, self.num_nodes)
        if self.demand_matrix.shape != expected_matrix_shape:
            raise ValueError("The demand matrix must be square and match the node count.")
        if self.distance_matrix.shape != expected_matrix_shape:
            raise ValueError("The distance matrix must match the node count.")
        if len(self.hub_costs) != self.num_nodes:
            raise ValueError("There must be one hub cost per node.")
        if len(self.capacity_limits) != self.num_nodes:
            raise ValueError("There must be one capacity limit per node.")
        if not 0 <= self.alpha <= 1:
            raise ValueError("The inter-hub discount factor alpha must be between 0 and 1.")
        if not 1 <= self.max_hubs <= self.num_nodes:
            raise ValueError("max_hubs must be between 1 and the number of nodes.")

    def geographic_clustering(self, num_clusters):
        """Partition node coordinates into reproducible geographic clusters."""
        if not 1 <= num_clusters <= min(self.max_hubs, self.num_nodes):
            raise ValueError("num_clusters must be between 1 and min(max_hubs, num_nodes).")

        kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(self.coordinates)
        clusters = {cluster_id: [] for cluster_id in range(num_clusters)}
        for node_id, cluster_id in enumerate(cluster_labels):
            clusters[int(cluster_id)].append(node_id)

        return clusters, kmeans.cluster_centers_

    def find_optimal_cluster_hub(self, cluster_nodes):
        """Enumerate feasible hub candidates and return the least-cost one."""
        cluster_flow = sum(
            self.demand_matrix[origin, destination]
            for origin in cluster_nodes
            for destination in cluster_nodes
            if origin != destination
        )
        best_hub = None
        best_cost = float("inf")

        for hub_candidate in cluster_nodes:
            if cluster_flow > self.capacity_limits[hub_candidate]:
                continue

            intra_cost = 0.0
            for origin in cluster_nodes:
                for destination in cluster_nodes:
                    if origin == destination:
                        continue
                    flow = self.demand_matrix[origin, destination]
                    route_distance = (
                        self.distance_matrix[origin, hub_candidate]
                        + self.alpha
                        * self.distance_matrix[hub_candidate, destination]
                    )
                    intra_cost += flow * route_distance

            total_cost = self.hub_costs[hub_candidate] + intra_cost
            if total_cost < best_cost:
                best_hub = hub_candidate
                best_cost = total_cost

        return best_hub, best_cost

    def calculate_inter_cluster_cost(
        self, hub1, hub2, cluster1_nodes, cluster2_nodes
    ):
        """Price directed traffic in both directions between two clusters."""
        inter_cost = 0.0

        for origin in cluster1_nodes:
            for destination in cluster2_nodes:
                flow = self.demand_matrix[origin, destination]
                route_distance = (
                    self.distance_matrix[origin, hub1]
                    + self.alpha * self.distance_matrix[hub1, hub2]
                    + self.alpha * self.distance_matrix[hub2, destination]
                )
                inter_cost += flow * route_distance

        for origin in cluster2_nodes:
            for destination in cluster1_nodes:
                flow = self.demand_matrix[origin, destination]
                route_distance = (
                    self.distance_matrix[origin, hub2]
                    + self.alpha * self.distance_matrix[hub2, hub1]
                    + self.alpha * self.distance_matrix[hub1, destination]
                )
                inter_cost += flow * route_distance

        return inter_cost

    def solve(self, num_clusters=None):
        """Run geographic clustering, hub selection, and cost calculation."""
        start_time = time.perf_counter()
        if num_clusters is None:
            num_clusters = min(self.max_hubs, self.num_nodes // 2)

        clusters, cluster_centers = self.geographic_clustering(num_clusters)
        selected_hubs = {}
        cluster_costs = {}
        infeasible_clusters = []

        for cluster_id, cluster_nodes in clusters.items():
            hub, cost = self.find_optimal_cluster_hub(cluster_nodes)
            if hub is None:
                infeasible_clusters.append(cluster_id)
            else:
                selected_hubs[cluster_id] = hub
                cluster_costs[cluster_id] = cost

        if infeasible_clusters:
            return {
                "feasible": False,
                "clusters": clusters,
                "selected_hubs": selected_hubs,
                "assignments": {},
                "cluster_centers": cluster_centers,
                "total_cost": None,
                "cost_breakdown": None,
                "computation_time": time.perf_counter() - start_time,
                "inter_cluster_costs": {},
                "infeasible_clusters": infeasible_clusters,
            }

        inter_cluster_costs = {}
        cluster_ids = list(clusters)
        for index, cluster1 in enumerate(cluster_ids):
            for cluster2 in cluster_ids[index + 1 :]:
                inter_cluster_costs[(cluster1, cluster2)] = (
                    self.calculate_inter_cluster_cost(
                        selected_hubs[cluster1],
                        selected_hubs[cluster2],
                        clusters[cluster1],
                        clusters[cluster2],
                    )
                )

        fixed_cost = sum(
            self.hub_costs[hub] for hub in selected_hubs.values()
        )
        intra_cluster_cost = sum(
            cluster_costs[cluster_id] - self.hub_costs[selected_hubs[cluster_id]]
            for cluster_id in clusters
        )
        inter_cluster_cost = sum(inter_cluster_costs.values())
        total_cost = fixed_cost + intra_cluster_cost + inter_cluster_cost

        assignments = {
            node: selected_hubs[cluster_id]
            for cluster_id, cluster_nodes in clusters.items()
            for node in cluster_nodes
        }
        assert set(assignments) == set(range(self.num_nodes))
        assert len(selected_hubs) <= self.max_hubs

        return {
            "feasible": True,
            "clusters": clusters,
            "selected_hubs": selected_hubs,
            "assignments": assignments,
            "cluster_centers": cluster_centers,
            "total_cost": total_cost,
            "cost_breakdown": {
                "fixed_cost": fixed_cost,
                "intra_cluster_cost": intra_cluster_cost,
                "inter_cluster_cost": inter_cluster_cost,
            },
            "computation_time": time.perf_counter() - start_time,
            "inter_cluster_costs": inter_cluster_costs,
            "infeasible_clusters": [],
        }


def calculate_distance_matrix(coords):
    """Calculate pairwise Euclidean distances with the example's scale factor."""
    coords = np.asarray(coords, dtype=float)
    differences = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return np.linalg.norm(differences, axis=2) * 50


# Node coordinates in two-dimensional geographic space.
coordinates = np.array(
    [
        [1, 1],
        [2, 1],
        [1, 2],
        [2, 2],
        [7, 1],
        [8, 1],
        [7, 2],
        [8, 2],
    ]
)
num_nodes = len(coordinates)

# Directed origin-destination demand; diagonal entries represent no self-demand.
demand_matrix = np.array(
    [
        [0, 15, 20, 12, 8, 10, 6, 9],
        [12, 0, 18, 25, 14, 7, 11, 8],
        [22, 16, 0, 15, 9, 12, 8, 13],
        [10, 28, 14, 0, 18, 15, 20, 11],
        [7, 11, 8, 16, 0, 22, 18, 25],
        [9, 6, 14, 11, 24, 0, 19, 28],
        [5, 9, 7, 19, 16, 17, 0, 21],
        [8, 7, 12, 9, 23, 26, 20, 0],
    ]
)

distance_matrix = calculate_distance_matrix(coordinates)
hub_costs = [800, 900, 750, 850, 1000, 1100, 950, 1050]
capacity_limits = [120, 140, 110, 130, 160, 170, 150, 165]

dc_solver = DivideConquerHubSpoke(
    coordinates=coordinates,
    demand_matrix=demand_matrix,
    distance_matrix=distance_matrix,
    hub_costs=hub_costs,
    capacity_limits=capacity_limits,
    alpha=0.75,
    max_hubs=3,
)

print("Eight-node network configuration")
print(f"Node coordinates: {coordinates.tolist()}")
print(f"Total directed demand: {demand_matrix.sum():.0f}")
positive_distances = distance_matrix[distance_matrix > 0]
print(f"Mean nonzero distance: {positive_distances.mean():.1f} distance units")
print(f"Maximum hubs allowed: {dc_solver.max_hubs}")

In [ ]:
solution = dc_solver.solve(num_clusters=2)

print("Divide-and-conquer heuristic")
print(f"Computation time: {solution['computation_time']:.6f} seconds")

if not solution["feasible"]:
    print("No feasible network was found under the stated cluster-capacity rule.")
    for cluster_id in solution["infeasible_clusters"]:
        nodes = [node + 1 for node in solution["clusters"][cluster_id]]
        print(f"Cluster {cluster_id} (nodes {nodes}) has no feasible hub candidate.")
else:
    print(f"Total network cost: ${solution['total_cost']:,.2f}")
    print("\nCluster formation and selected hubs")
    for cluster_id, nodes in solution["clusters"].items():
        hub = solution["selected_hubs"][cluster_id]
        print(f"Cluster {cluster_id}: nodes {[n + 1 for n in nodes]} -> hub {hub + 1}")

    print("\nCost breakdown")
    for name, cost in solution["cost_breakdown"].items():
        print(f"{name.replace('_', ' ').title()}: ${cost:,.2f}")

    print("\nSpoke assignments")
    for node, hub in solution["assignments"].items():
        if node == hub:
            print(f"Node {node + 1} is a hub.")
        else:
            print(f"Node {node + 1} -> hub {hub + 1}")

    calculated_total = sum(solution["cost_breakdown"].values())
    assert np.isclose(calculated_total, solution["total_cost"])

In [ ]:
def visualize_divide_conquer_solution(dc_solver, solution):
    """Plot clusters, selected hubs, and the cost and cluster-size summaries."""
    if not solution["feasible"]:
        print("Visualization of a feasible network is unavailable because no feasible hub selection was found.")
        return None

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Geographic Hub-and-Spoke Network Design", fontsize=16, fontweight="bold")
    colors = ["#FF6B6B", "#4ECDC4", "#FFD93D", "#95E77E"]

    ax1.set_title("Geographic Clusters and Hubs", fontweight="bold")
    for cluster_id, nodes in solution["clusters"].items():
        color = colors[cluster_id % len(colors)]
        hub = solution["selected_hubs"][cluster_id]
        for node in nodes:
            x_coord, y_coord = dc_solver.coordinates[node]
            is_hub = node == hub
            ax1.scatter(
                x_coord,
                y_coord,
                s=280 if is_hub else 150,
                c=color,
                marker="s" if is_hub else "o",
                alpha=1.0 if is_hub else 0.7,
                edgecolors="black",
                linewidth=1.5,
                label=f"Cluster {cluster_id}; hub" if is_hub else None,
            )
            ax1.text(x_coord + 0.08, y_coord + 0.08, str(node + 1), fontweight="bold")
            if not is_hub:
                hub_x, hub_y = dc_solver.coordinates[hub]
                ax1.plot([x_coord, hub_x], [y_coord, hub_y], "--", color=color, alpha=0.7)

    hubs = list(solution["selected_hubs"].values())
    for index, hub1 in enumerate(hubs):
        for hub2 in hubs[index + 1 :]:
            x1, y1 = dc_solver.coordinates[hub1]
            x2, y2 = dc_solver.coordinates[hub2]
            ax1.plot([x1, x2], [y1, y2], color="black", linewidth=2, alpha=0.8)
    ax1.set_xlabel("X coordinate (coordinate units)")
    ax1.set_ylabel("Y coordinate (coordinate units)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.set_title("Cost Breakdown", fontweight="bold")
    breakdown = solution["cost_breakdown"]
    categories = ["Fixed hubs", "Within-cluster", "Between-cluster"]
    costs = [
        breakdown["fixed_cost"],
        breakdown["intra_cluster_cost"],
        breakdown["inter_cluster_cost"],
    ]
    bars = ax2.bar(categories, costs, color=["#FF6B6B", "#4ECDC4", "#FFD93D"], alpha=0.85)
    ax2.set_ylabel("Cost (currency units)")
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, cost in zip(bars, costs):
        ax2.annotate(
            f"${cost:,.0f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
        )

    ax3.set_title("Solution Summary", fontweight="bold")
    summary_labels = ["Clusters", "Selected hubs"]
    summary_values = [len(solution["clusters"]), len(solution["selected_hubs"])]
    bars = ax3.bar(summary_labels, summary_values, color="#95E77E", alpha=0.85)
    ax3.set_ylabel("Count")
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, summary_values):
        ax3.annotate(
            str(value),
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
        )

    ax4.set_title("Cluster Composition", fontweight="bold")
    cluster_ids = list(solution["clusters"])
    cluster_sizes = [len(solution["clusters"][cluster_id]) for cluster_id in cluster_ids]
    bars = ax4.bar(
        [f"Cluster {cluster_id}" for cluster_id in cluster_ids],
        cluster_sizes,
        color=[colors[cluster_id % len(colors)] for cluster_id in cluster_ids],
        alpha=0.85,
    )
    ax4.set_xlabel("Cluster")
    ax4.set_ylabel("Number of nodes")
    ax4.grid(True, axis="y", alpha=0.3)
    for bar, cluster_id in zip(bars, cluster_ids):
        hub = solution["selected_hubs"][cluster_id]
        ax4.annotate(
            f"Hub {hub + 1}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
        )

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_divide_conquer_solution(dc_solver, solution)

## Small-instance reference search

The following exhaustive enumeration is a diagnostic reference for a *different, simplified model*: it selects a set of hubs and routes each directed origin-destination flow through its least-cost single hub. Its objective does not include the cluster-to-cluster hub-to-hub route used by the divide-and-conquer heuristic. Therefore, its objective values are not an optimality benchmark for the full clustered network model, and a cost gap between the two models would not have a meaningful interpretation.

The enumeration is retained to show how candidate hub sets grow combinatorially. For \(n\) nodes and at most \(p\) hubs, the number of hub sets examined is \(\sum_{r=1}^{p}\binom{n}{r}\). In the reference calculation, each selected hub is tested against the entire network's demand under the stated capacity limit. The function reports when no hub set satisfies that simplified capacity rule rather than returning a fabricated optimum.

In [ ]:
def compare_with_exhaustive_search(dc_solver, max_nodes_for_exhaustive=6):
    """Enumerate hub sets for the explicitly simplified single-hub routing model."""
    if dc_solver.num_nodes > max_nodes_for_exhaustive:
        return {
            "feasible": False,
            "reason": (
                f"Network has {dc_solver.num_nodes} nodes; the reference limit is "
                f"{max_nodes_for_exhaustive}."
            ),
        }

    total_flow = float(dc_solver.demand_matrix.sum())
    best_cost = float("inf")
    best_solution = None

    for num_hubs in range(1, min(dc_solver.max_hubs, dc_solver.num_nodes) + 1):
        for hub_combination in combinations(range(dc_solver.num_nodes), num_hubs):
            if any(
                total_flow > dc_solver.capacity_limits[hub]
                for hub in hub_combination
            ):
                continue

            fixed_cost = sum(dc_solver.hub_costs[hub] for hub in hub_combination)
            variable_cost = 0.0
            for origin in range(dc_solver.num_nodes):
                for destination in range(dc_solver.num_nodes):
                    if origin == destination:
                        continue
                    flow = dc_solver.demand_matrix[origin, destination]
                    least_cost = min(
                        flow
                        * (
                            dc_solver.distance_matrix[origin, hub]
                            + dc_solver.alpha
                            * dc_solver.distance_matrix[hub, destination]
                        )
                        for hub in hub_combination
                    )
                    variable_cost += least_cost

            total_cost = fixed_cost + variable_cost
            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = {
                    "hubs": list(hub_combination),
                    "cost": total_cost,
                }

    if best_solution is None:
        return {
            "feasible": False,
            "reason": "No enumerated hub set satisfies the reference capacity rule.",
        }

    return {"feasible": True, **best_solution}


print("Small-instance reference search")
test_sizes = [4, 5, 6]
reference_results = []

for size in test_sizes:
    coords_subset = coordinates[:size]
    demand_subset = demand_matrix[:size, :size]
    dist_subset = calculate_distance_matrix(coords_subset)
    solver_subset = DivideConquerHubSpoke(
        coords_subset,
        demand_subset,
        dist_subset,
        hub_costs[:size],
        capacity_limits[:size],
        alpha=0.75,
        max_hubs=2,
    )
    start_time = time.perf_counter()
    reference_solution = compare_with_exhaustive_search(solver_subset)
    elapsed_time = time.perf_counter() - start_time

    if reference_solution["feasible"]:
        reference_results.append(
            {
                "size": size,
                "cost": reference_solution["cost"],
                "time": elapsed_time,
            }
        )
        print(
            f"Size {size}: reference cost = ${reference_solution['cost']:,.2f}; "
            f"enumeration time = {elapsed_time:.6f} seconds."
        )
    else:
        print(f"Size {size}: {reference_solution['reason']}")

if reference_results:
    fig, ax = plt.subplots(figsize=(8, 5))
    sizes = [result["size"] for result in reference_results]
    costs = [result["cost"] for result in reference_results]
    ax.plot(sizes, costs, "o-", color="#4ECDC4", linewidth=2, markersize=7)
    ax.set_title("Simplified Reference-Model Cost by Instance Size")
    ax.set_xlabel("Number of nodes")
    ax.set_ylabel("Cost (currency units)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No feasible reference-model instances are available for plotting.")

## Scalability experiment

This experiment generates reproducible random instances at several network sizes and records the heuristic's runtime and cost when the stated capacity rule admits a feasible solution. A fixed random seed makes each generated instance reproducible. The experiment does not establish an asymptotic complexity guarantee: measured runtime depends on the data, software environment, and hardware, while the plotted \(n\log n\) curve is only a visual reference and is not the algorithm's derived complexity.

In [ ]:
def scalability_analysis():
    """Measure heuristic runtime and cost on reproducible random instances."""
    network_sizes = [8, 12, 16, 20, 24]
    computation_times = []
    solution_costs = []
    feasible_sizes = []
    rng = np.random.RandomState(42)

    for size in network_sizes:
        coords = rng.rand(size, 2) * 10
        demand = rng.randint(5, 30, (size, size))
        np.fill_diagonal(demand, 0)
        dist = calculate_distance_matrix(coords)
        random_hub_costs = rng.randint(800, 1200, size)
        capacities = rng.randint(100, 200, size)
        solver = DivideConquerHubSpoke(
            coords,
            demand,
            dist,
            random_hub_costs,
            capacities,
            alpha=0.75,
            max_hubs=min(3, size // 3),
        )

        start_time = time.perf_counter()
        result = solver.solve()
        elapsed_time = time.perf_counter() - start_time

        if result["feasible"]:
            feasible_sizes.append(size)
            computation_times.append(elapsed_time)
            solution_costs.append(result["total_cost"])
            print(
                f"Size {size:2d}: time = {elapsed_time:.6f} seconds; "
                f"cost = ${result['total_cost']:,.2f}."
            )
        else:
            print(
                f"Size {size:2d}: no feasible solution under the stated "
                "cluster-capacity rule."
            )

    if feasible_sizes:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Heuristic Scalability Experiment", fontweight="bold")

        ax1.plot(
            feasible_sizes,
            computation_times,
            "o-",
            color="#FF6B6B",
            linewidth=2,
            markersize=7,
            label="Measured runtime",
        )
        reference = np.asarray(feasible_sizes, dtype=float) * np.log(feasible_sizes)
        if computation_times[0] > 0:
            reference *= computation_times[0] / reference[0]
            ax1.plot(
                feasible_sizes,
                reference,
                "--",
                color="gray",
                label="Scaled n log(n) visual reference",
            )
        ax1.set_title("Runtime and Network Size")
        ax1.set_xlabel("Number of nodes")
        ax1.set_ylabel("Runtime (seconds)")
        ax1.set_yscale("log")
        ax1.grid(True, alpha=0.3)
        ax1.legend()

        cost_per_node = [
            cost / size for cost, size in zip(solution_costs, feasible_sizes)
        ]
        ax2.plot(
            feasible_sizes,
            cost_per_node,
            "o-",
            color="#4ECDC4",
            linewidth=2,
            markersize=7,
        )
        ax2.set_title("Cost per Node")
        ax2.set_xlabel("Number of nodes")
        ax2.set_ylabel("Cost per node (currency units)")
        ax2.grid(True, alpha=0.3)

        fig.tight_layout()
        plt.show()
    else:
        print("No feasible instances are available for the scalability plots.")

    return feasible_sizes, computation_times, solution_costs


sizes, times, costs = scalability_analysis()

## When this heuristic is useful

Geographic clustering can provide a practical way to organize hub-location decisions when nearby nodes are expected to share service regions. Its value depends on whether geographic proximity is a useful proxy for the network's actual demand and cost structure. The procedure can be useful for producing a structured candidate design, but the clustering choice can exclude better assignments, and the method has no global optimality guarantee.

### Strengths and limitations

**Strengths**

- Geographic clustering divides hub selection into smaller candidate searches.
- Hub candidates are evaluated with directed demand and the specified routing-cost formula.
- Fixed, within-cluster, and between-cluster costs are reported separately.
- Capacity infeasibility is reported explicitly rather than hidden by an arbitrary fallback hub.

**Limitations**

- K-means optimizes geometric clustering criteria, not the complete logistics objective.
- The selected partition can constrain which hubs and assignments are considered.
- The stated capacity rule counts within-cluster directed demand only; it does not model total physical hub throughput, inter-cluster transshipment capacity, or congestion.
- The heuristic does not guarantee a globally optimal solution, and its quality cannot be summarized by a universal percentage gap.
- Runtime measurements from small generated examples are empirical observations, not complexity proofs or guarantees of speedup over an exact method.

### Further modeling considerations

A practical deployment may require additional assumptions or constraints, such as throughput definitions that include inter-cluster flows, service-level requirements, facility eligibility, or transport-mode limits. Such additions change the modeled problem and should be defined consistently in both the mathematical formulation and the implementation.